In [ ]:
import pandas as pd

# 读取CSV文件，只加载需要的列
df = pd.read_csv('Aventa_AV7_IET_OST_SCADA.csv', usecols=['Datetime', 'WindSpeed'])

# 转换Datetime列为datetime类型并设为索引
df['Datetime'] = pd.to_datetime(df['Datetime'])
df.set_index('Datetime', inplace=True)

# 按10分钟聚合（计算平均值）
df_10min = df.resample('10T').mean().reset_index()

# 按1分钟聚合（计算平均值）
df_1min = df.resample('1T').mean().reset_index()

# 保存结果到CSV文件
df_10min.to_csv('Aventa_AV7_IET_OST_SCADA_10min_mean.csv', index=False)
df_1min.to_csv('Aventa_AV7_IET_OST_SCADA_1min_mean.csv', index=False)

print("处理完成！已生成两个聚合文件。")

In [ ]:
import pandas as pd

# 保存结果到CSV文件
df_10min=pd.read_csv('Aventa_AV7_IET_OST_SCADA_10min_mean.csv')
df_1min=pd.read_csv('Aventa_AV7_IET_OST_SCADA_1min_mean.csv')
df_10min.head()

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.stats import norm

# 确保时间列是datetime类型
df_10min['Datetime'] = pd.to_datetime(df_10min['Datetime'])

# 创建1分钟间隔的时间序列
start_time = df_10min['Datetime'].min()
end_time = df_10min['Datetime'].max()
time_1min = pd.date_range(start=start_time, end=end_time, freq='1min')

# 线性插值（所有站点同时插值）
df_indexed = df_10min.set_index('Datetime')
df_1min_indexed = df_indexed.reindex(time_1min)
df_1min_indexed[['WindSpeed']] = df_indexed[['WindSpeed']].reindex(time_1min).interpolate(method='linear')

# 重置索引并保存为CSV（无行数限制)
df_1min_inter = df_1min_indexed.reset_index().rename(columns={'index': 'Datetime'})
df_1min_inter.head()

In [ ]:
df_original = df_10min
# -------------------------- 2. 配置波动参数 --------------------------
sites = ['WindSpeed']
noise_params = {
    'white_noise_std': 0.1,       # 高频白噪声标准差（占原始风速的比例，如0.2表示20%波动）
    'diurnal_amplitude': 0.1,     # 日周期波动幅度（原始风速的10%）
    'clip_min': 0                 # 风速最小值（避免负数）
}

# -------------------------- 3. 生成复合随机波动 --------------------------
def generate_wind_noise(df, sites, params):
    """
    生成符合风速特性的复合随机波动：
    - 高频白噪声（短时间尺度随机起伏）
    - 低频日周期波动（缓慢的日变化）
    """
    n = len(df)
    t = np.arange(n)  # 时间索引（分钟）
    
    # 1. 高频白噪声（每个站点独立生成）
    white_noise = np.zeros((n, len(sites)))
    for i, site in enumerate(sites):
        # 白噪声标准差：基于原始数据的标准差调整
        site_std = df_original[site].std()
        white_noise[:, i] = norm.rvs(
            loc=0,
            scale=site_std * params['white_noise_std'],
            size=n
        )
    
    # 2. 低频日周期波动（所有站点共享日趋势）
    diurnal_cycle = np.sin(2 * np.pi * t / 1440)  # 1440分钟=1天，周期为1天
    diurnal_noise = diurnal_cycle.reshape(-1, 1) * params['diurnal_amplitude']
    
    # 3. 合成总波动（高频+低频）
    total_noise = white_noise + diurnal_noise

    for i in range(len(total_noise)):
        if i % 10 == 0:
            total_noise[i]=0
        else:
            None
    return total_noise

# 生成波动
noise = generate_wind_noise(df_1min, sites, noise_params)

# -------------------------- 4. 叠加波动并修正数据 --------------------------
df_noisy = df_1min.copy()
for i, site in enumerate(sites):
    # 叠加波动
    df_noisy[site] = df_noisy[site] + noise[:, i]
.head()

In [ ]:
df_noisy = df_noisy.rename(columns={'WindSpeed': 'WindSpeed_noise'})
df_1min['Datetime'] = pd.to_datetime(df_1min['Datetime'])
df_noisy['Datetime'] = pd.to_datetime(df_noisy['Datetime'])
# 按Datetime列合并两个DataFrame
merged_df = pd.merge(
    df_1min, 
    df_noisy, 
    on='Datetime', 
    how='left'  # 使用左连接保留df_1min的所有时间点
)
merged_df =merged_df.fillna(0)
# 查看合并结果
print(merged_df.head())

In [ ]:
df_1min_inter = df_1min_inter.rename(columns={'WindSpeed': 'WindSpeed_no_noise'})
# 按Datetime列合并两个DataFrame
merged_df = pd.merge(
    merged_df, 
    df_1min_inter, 
    on='Datetime', 
    how='left'  # 使用左连接保留df_1min的所有时间点
)
merged_df =merged_df.fillna(0)
# 查看合并结果
print(merged_df.head())

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd
import matplotlib.dates as mdates
# 设置中文显示
plt.rcParams["font.family"] = ["Heiti TC"]
plt.rcParams['axes.unicode_minus'] = False  # 解决负号显示问题

# 确保Datetime列为datetime类型
merged_df['Datetime'] = pd.to_datetime(merged_df['Datetime'])

# 删除缺失数据行
clean_df = merged_df.dropna()
clean_df = clean_df.iloc[-2500:,:]

# 创建图形和坐标轴
fig, ax = plt.subplots(figsize=(14, 7))

# 绘制WindSpeed（灰色，透明度0.5）
ax.plot(clean_df['Datetime'], clean_df['WindSpeed'], 
        color='#808080', alpha=0.5, linewidth=1.5, label='WindSpeed')

# 绘制WindSpeed_noise（蓝色实线）
ax.plot(clean_df['Datetime'], clean_df['WindSpeed_noise'], 
        color='#1E88E5', linewidth=2, label='WindSpeed_noise')

# 设置图形属性
ax.set_xlabel('时间', fontsize=12)
ax.set_ylabel('风速值', fontsize=12)
ax.set_title('风速与风速噪声对比图', fontsize=16, fontweight='bold', pad=20)

# 设置网格
ax.grid(True, alpha=0.3, linestyle='--')

# 设置图例
ax.legend(loc='upper right', fontsize=11, framealpha=0.9)

# 优化x轴时间刻度
if len(clean_df) > 0:
    # 根据数据时间范围自动设置刻度
    time_range = clean_df['Datetime'].max() - clean_df['Datetime'].min()
    
    if time_range.days > 1:
        # 如果时间跨度大于1天，按天显示
        ax.xaxis.set_major_locator(mdates.DayLocator())
        ax.xaxis.set_major_formatter(mdates.DateFormatter('%Y-%m-%d'))
    else:
        # 如果时间跨度小于1天，按小时显示
        ax.xaxis.set_major_locator(mdates.HourLocator())
        ax.xaxis.set_major_formatter(mdates.DateFormatter('%H:%M'))
    
    # 旋转x轴标签
    plt.setp(ax.xaxis.get_majorticklabels(), rotation=45)

# 添加背景色
ax.set_facecolor('#f8f9fa')
fig.patch.set_facecolor('white')

# 自动调整布局
plt.tight_layout()

# 显示图形
plt.show()

# 可选：保存图片
# plt.savefig('windspeed_comparison.png', dpi=300, bbox_inches='tight')

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
from sklearn.metrics import mean_squared_error, mean_absolute_error

# 假设你的数据已经加载为DataFrame df
# 如果不是，请使用：df = pd.read_csv('your_data.csv')

df = merged_df.iloc[-2500:,:]

# 设置全局字体大小
plt.rcParams.update({
    'font.size': 16,           # 基础字体大小
    'axes.titlesize': 18,      # 标题字体大小
    'axes.labelsize': 16,      # 坐标轴标签字体大小
    'xtick.labelsize': 14,     # X轴刻度标签字体大小
    'ytick.labelsize': 14,     # Y轴刻度标签字体大小
    'legend.fontsize': 14,     # 图例字体大小
    'figure.titlesize': 20     # 图形标题字体大小
})

# 1. 时间序列对比图 - 直观展示预测值与真实值的匹配程度
fig = plt.figure(figsize=(20, 10))

# 子图1: 完整时间序列对比
ax1 = plt.subplot(4, 1, 1)
ax1.plot(df['Datetime'], df['WindSpeed'], 'k-', linewidth=2, label='True Value', alpha=0.8)
ax1.plot(df['Datetime'], df['WindSpeed_noise'], 'r--', linewidth=1.5, label='ARNN', alpha=0.7)
ax1.plot(df['Datetime'], df['WindSpeed_no_noise'], 'b-.', linewidth=1.5, label='linear Interpolation', alpha=0.7)
ax1.set_ylabel('Wind Speed', fontsize=16)
ax1.set_title('Time Series Comparison', fontsize=18, pad=20)
ax1.legend(loc='best', fontsize=14)
ax1.grid(True, alpha=0.3)
# 去除X轴刻度标签和标题
ax1.set_xticklabels([])
ax1.set_xlabel('')

# 子图3: 预测误差时间序列
ax3 = plt.subplot(4, 1, 2)
error_noise = df['WindSpeed_noise'] - df['WindSpeed']
error_no_noise = df['WindSpeed_no_noise'] - df['WindSpeed']
ax3.plot(df['Datetime'], error_noise, 'r-', linewidth=1, label='Error (ARNN)', alpha=0.6)
ax3.plot(df['Datetime'], error_no_noise, 'b-', linewidth=1, label='Error (linear Interpolation)', alpha=0.6)
ax3.axhline(y=0, color='k', linestyle='-', linewidth=0.5, alpha=0.5)
ax3.fill_between(df['Datetime'], error_noise, 0, alpha=0.2, color='red')
ax3.fill_between(df['Datetime'], error_no_noise, 0, alpha=0.2, color='blue')
ax3.set_ylabel('Prediction Error', fontsize=16)
ax3.set_title('Prediction Errors Over Time', fontsize=18, pad=20)
ax3.legend(loc='best', fontsize=14)
ax3.grid(True, alpha=0.3)
# 去除X轴刻度标签和标题
ax3.set_xticklabels([])
ax3.set_xlabel('')

# 子图4: 累积误差图
ax4 = plt.subplot(4, 1, 3)
# 计算累积绝对误差
cumulative_error_noise = np.cumsum(np.abs(error_noise))
cumulative_error_no_noise = np.cumsum(np.abs(error_no_noise))

ax4.plot(df['Datetime'], cumulative_error_noise, 'r-', linewidth=2, label='Cumulative Absolute Error (ARNN)')
ax4.plot(df['Datetime'], cumulative_error_no_noise, 'b-', linewidth=2, label='Cumulative Absolute Error (linear Interpolation)')
ax4.set_ylabel('Cumulative Absolute Error', fontsize=16)
ax4.set_title('Cumulative Absolute Error Over Time', fontsize=18, pad=20)
ax4.legend(loc='best', fontsize=14)
ax4.grid(True, alpha=0.3)


# 子图2: 放大查看细节（前50个点）
ax2 = plt.subplot(4, 1, 4)
n_points = min(50, len(df))
ax2.plot(df['Datetime'].iloc[:n_points], df['WindSpeed'].iloc[:n_points], 'k-', linewidth=2, label='True Value', marker='o', markersize=4)
ax2.plot(df['Datetime'].iloc[:n_points], df['WindSpeed_noise'].iloc[:n_points], 'r--', linewidth=1.5, label='ARNN', marker='s', markersize=4)
ax2.plot(df['Datetime'].iloc[:n_points], df['WindSpeed_no_noise'].iloc[:n_points], 'b-.', linewidth=1.5, label='linear Interpolation', marker='^', markersize=4)
ax2.set_ylabel('Wind Speed', fontsize=16)
ax2.set_title(f'Zoomed View (First {n_points} Points)', fontsize=18, pad=20)
ax2.legend(loc='best', fontsize=14)
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

# 4. 性能指标汇总表
print("="*60)
print("PERFORMANCE METRICS COMPARISON")
print("="*60)

# 计算各种评估指标
metrics = {
    'MSE (Mean Squared Error)': [mean_squared_error(df['WindSpeed'], df['WindSpeed_noise']),
                                  mean_squared_error(df['WindSpeed'], df['WindSpeed_no_noise'])],
    'RMSE (Root Mean Squared Error)': [np.sqrt(mean_squared_error(df['WindSpeed'], df['WindSpeed_noise'])),
                                       np.sqrt(mean_squared_error(df['WindSpeed'], df['WindSpeed_no_noise']))],
    'MAE (Mean Absolute Error)': [mean_absolute_error(df['WindSpeed'], df['WindSpeed_noise']),
                                  mean_absolute_error(df['WindSpeed'], df['WindSpeed_no_noise'])],
    'Mean Error': [error_noise.mean(), error_no_noise.mean()],
    'Std of Error': [error_noise.std(), error_no_noise.std()],
    'R² Score': [np.corrcoef(df['WindSpeed'], df['WindSpeed_noise'])[0,1]**2,
                 np.corrcoef(df['WindSpeed'], df['WindSpeed_no_noise'])[0,1]**2]
}

# 创建并显示指标表格
metrics_df = pd.DataFrame(metrics, index=['With Noise', 'Without Noise'])
print(metrics_df.round(4))
print("\n" + "="*60)



In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import os

try:
    # 读取CSV文件
    df = merged_df.copy()
    
    # 打印前5行，检查数据格式
    print("数据前5行：")
    print(df.head())
    
    # 将第一列转换为时间序列
    df.iloc[:, 0] = pd.to_datetime(df.iloc[:, 0], errors='coerce')
    
    # 检查时间转换是否成功
    if df.iloc[:, 0].isna().any():
        print("警告：部分时间数据无法解析，可能格式不一致！")
    
    # 提取时间和数据
    time = df.iloc[:, 0]
    data = pd.to_numeric(df.iloc[:, 2], errors='coerce')  # 确保第二列是数值
    
    # 检查数据是否有效
    if data.isna().any():
        print("警告：数据列中存在无效值（非数值），已转换为NaN！")
    
    # 计算布林通道（周期=100，k=2）
    window = 500
    k = 1
    # 中轨：100周期移动平均
    middle_band = data.rolling(window=window).mean()
    # 标准差
    std = data.rolling(window=window).std()
    # 上轨和下轨
    upper_band = middle_band + k * std
    lower_band = middle_band - k * std
    
    # 绘制折线图和布林通道
    plt.figure(figsize=(12, 6))  # 增大画布，适应大数据
    # 绘制原始数据
    plt.plot(time, data, color='blue', label='Wind Data', linewidth=1)
    # 绘制布林通道
    plt.plot(time, middle_band, color='orange', label='Middle Band (SMA 100)', linewidth=1)
    plt.plot(time, upper_band, color='green', label='Upper Band', linewidth=1)
    plt.plot(time, lower_band, color='red', label='Lower Band', linewidth=1)
    # 填充上轨和下轨之间的区域
    plt.fill_between(time, lower_band, upper_band, color='gray', alpha=0.2, label='Bollinger Band')
    
    # 设置Y轴标签和标题
    plt.ylabel('Data Value')
    plt.title('1-Minute Wind Data with Bollinger Bands (Period=100)')
    plt.legend()
    plt.grid(True)
    
    # 移除X轴标签和刻度
    plt.gca().set_xticks([])  # 不显示X轴刻度
    plt.gca().set_xlabel('')  # 移除X轴标题
    
    # 优化布局
    plt.tight_layout()
    
    # 保存图片（防止显示卡顿）
    plt.savefig('wind_plot_with_bollinger.png', dpi=100)
    print("图表已保存为 wind_plot_with_bollinger.png，请检查当前目录！")
    
    # 显示图形（如果环境支持）
    plt.show()

except Exception as e:
    print(f"出错啦：{e}")

In [ ]:
# 统计数值大于上轨或小于下轨的点
outliers = data[(data > upper_band) | (data < lower_band)]
outlier_count = len(outliers)
outlier_percentage = (outlier_count / len(data)) * 100
divisible_by_10 = outliers[outliers.index % 10 == 0]
index_list = divisible_by_10.index / 10
df_10min['index_flag'] = df_10min.index.isin(index_list).astype(int)
df_10min.to_excel('df_Aventa.xlsx')

In [ ]:
print(f"标志分布: 0-{np.sum(df_10min.index_flag == 0)}个, 1-{np.sum(df_10min.index_flag == 1)}个")